# Project 5 — Amazon Review Sentiment Classifier

Fine-tuned BERT base on Amazon product reviews to classify 
sentiment as Positive or Negative.

## Dataset
- Source: Amazon Polarity (HuggingFace)
- Training samples: 6,000
- Test samples: 1,000
- Class balance: 47% positive, 53% negative

## Results
| Epoch | Accuracy | F1 | Precision | Recall |
|---|---|---|---|---|
| 1 | 92.90% | 92.82% | 94.25% | 91.43% |
| 2 | 93.70% | 93.76% | 93.29% | 94.22% ← best |
| 3 | 93.70% | 93.71% | 93.99% | 93.43% |

Best model: Epoch 2 (load_best_model_at_end caught overfitting at Epoch 3)

## Key Findings

### What the model learned
- Dominant sentiment wins over mixed signals
- Last clause in a sentence carries more weight (recency bias)
- Product sentiment overrides delivery complaints
- Bad battery life dominates good design

### Word order sensitivity
- "terrible but I love" → POSITIVE (last clause wins)
- "I love but terrible" → NEGATIVE (last clause wins)
- Recency bias is a real limitation for mixed reviews

### Comparison to Project 2
| | Project 2 (Toxicity) | Project 5 (Sentiment) |
|---|---|---|
| Model | DistilBERT 67M | BERT base 110M |
| Dataset balance | 95/5 skewed | 50/50 balanced |
| Final F1 | 54.29% | 93.76% |
| Main challenge | Class imbalance | Recency bias |

## Tech Stack
- Model: bert-base-uncased (110M parameters)
- Framework: HuggingFace Transformers + Trainer API
- Dataset: amazon_polarity
- Metrics: Accuracy, F1, Precision, Recall


In [2]:
!pip install transformers datasets evaluate accelerate -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.0 MB/s eta 0:00:00


In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import TrainingArguments, Trainer
from datasets import load_dataset
import evaluate
import numpy as np

print("GPU available:", torch.cuda.is_available())
print("Libraries loaded successfully")


GPU available: True
Libraries loaded successfully


In [4]:
# Load Amazon reviews dataset
dataset = load_dataset("amazon_polarity")
print(dataset)
print("\nSample entry:")
print(dataset["train"][0])

README.md: 0.00B [00:00, ?B/s]

amazon_polarity/train-00000-of-00004.par(…):   0%|          | 0.00/260M [00:00<?, ?B/s]

amazon_polarity/train-00001-of-00004.par(…):   0%|          | 0.00/258M [00:00<?, ?B/s]

amazon_polarity/train-00002-of-00004.par(…):   0%|          | 0.00/255M [00:00<?, ?B/s]

amazon_polarity/train-00003-of-00004.par(…):   0%|          | 0.00/254M [00:00<?, ?B/s]

amazon_polarity/test-00000-of-00001.parq(…):   0%|          | 0.00/117M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3600000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/400000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['label', 'title', 'content'],
        num_rows: 3600000
    })
    test: Dataset({
        features: ['label', 'title', 'content'],
        num_rows: 400000
    })
})

Sample entry:
{'label': 1, 'title': 'Stuning even for the non-gamer', 'content': 'This sound track was beautiful! It paints the senery in your mind so well I would recomend it even to people who hate vid. game music! I have played the game Chrono Cross but out of all of the games I have ever played it has the best music! It backs away from crude keyboarding and takes a fresher step with grate guitars and soulful orchestras. It would impress anyone who cares to listen! ^_^'}


In [5]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# Use small subset
small_train = dataset["train"].select(range(6000))
small_test = dataset["test"].select(range(1000))

# Check class balance
positive = sum(1 for x in small_train if x["label"] == 1)
negative = sum(1 for x in small_train if x["label"] == 0)
print(f"Training set — Positive: {positive} | Negative: {negative}")

def preprocess(examples):
    # Combine title and content for richer input
    texts = [
        t + ". " + c 
        for t, c in zip(examples["title"], examples["content"])
    ]
    tokens = tokenizer(
        texts,
        truncation=True,
        padding="max_length",
        max_length=256
    )
    tokens["labels"] = examples["label"]
    return tokens

tokenized_train = small_train.map(preprocess, batched=True)
tokenized_test = small_test.map(preprocess, batched=True)

print("Train size:", len(tokenized_train))
print("Test size:", len(tokenized_test))
print("\nSample tokenized entry keys:", list(tokenized_train[0].keys()))

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Training set — Positive: 2822 | Negative: 3178


Map:   0%|          | 0/6000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Train size: 6000
Test size: 1000

Sample tokenized entry keys: ['label', 'title', 'content', 'input_ids', 'token_type_ids', 'attention_mask', 'labels']


In [6]:
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("Model loaded successfully")
print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Model type: {type(model).__name__}")

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded successfully
Total parameters:     109,483,778
Trainable parameters: 109,483,778
Model type: BertForSequenceClassification


In [7]:
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    
    accuracy = accuracy_metric.compute(
        predictions=predictions, references=labels)
    f1 = f1_metric.compute(
        predictions=predictions, references=labels, average="binary")
    
    tp = int(sum((p == 1 and l == 1) for p, l in zip(predictions, labels)))
    fp = int(sum((p == 1 and l == 0) for p, l in zip(predictions, labels)))
    fn = int(sum((p == 0 and l == 1) for p, l in zip(predictions, labels)))
    tn = int(sum((p == 0 and l == 0) for p, l in zip(predictions, labels)))
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    
    print(f"\n📊 Epoch Results:")
    print(f"   Accuracy:  {accuracy['accuracy']:.2%}")
    print(f"   F1:        {f1['f1']:.2%}")
    print(f"   Precision: {precision:.2%}")
    print(f"   Recall:    {recall:.2%}")
    print(f"   TP={tp} FP={fp} FN={fn} TN={tn}")
    
    return {
        **accuracy,
        **f1,
        "precision": precision,
        "recall": recall
    }

print("Metrics ready")

Metrics ready


In [8]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="amazon-review-classifier",
    eval_strategy="epoch",
    save_strategy="epoch",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    load_best_model_at_end=True,
    logging_steps=50,
    warmup_steps=200,
    weight_decay=0.01,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    compute_metrics=compute_metrics,
)

trainer.train()


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.582742,0.400262,0.929000,0.928210,0.942505,0.914343
2,0.339332,0.354070,0.937000,0.937562,0.932939,0.942231
3,0.218392,0.467791,0.937000,0.937063,0.939880,0.934263



📊 Epoch Results:
   Accuracy:  92.90%
   F1:        92.82%
   Precision: 94.25%
   Recall:    91.43%
   TP=459 FP=28 FN=43 TN=470


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


...

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]



📊 Epoch Results:
   Accuracy:  93.70%
   F1:        93.71%
   Precision: 93.99%
   Recall:    93.43%
   TP=469 FP=30 FN=33 TN=468


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=564, training_loss=0.46607652243147507, metrics={'train_runtime': 491.7501, 'train_samples_per_second': 36.604, 'train_steps_per_second': 1.147, 'total_flos': 2367999498240000.0, 'train_loss': 0.46607652243147507, 'epoch': 3.0})

In [9]:
import torch
import torch.nn.functional as F

device = next(model.parameters()).device

def predict_sentiment(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=256
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model(**inputs)
    
    probs = F.softmax(outputs.logits, dim=-1)
    predicted = torch.argmax(probs).item()
    confidence = probs[0][predicted].item()
    
    label = "⭐⭐⭐⭐⭐ POSITIVE" if predicted == 1 else "⭐ NEGATIVE"
    return label, confidence

# Test on real reviews
reviews = [
    # Clear positive
    "This product is absolutely amazing, works perfectly out of the box!",
    "Best purchase I have made this year. Highly recommend to everyone.",
    
    # Clear negative
    "Broke after two days. Complete waste of money. Do not buy.",
    "Terrible quality, nothing like the description. Very disappointed.",
    
    # Tricky cases
    "It's okay I guess, nothing special but does the job",
    "Not bad for the price but I expected better quality",
    "The product is good but delivery took forever",
    "I love the design but the battery life is terrible",
    
    # Your own test
    "This distributed systems book completely changed how I think about software",
]

print(f"{'Review':<55} {'Sentiment':<25} {'Confidence'}")
print("-" * 90)
for review in reviews:
    label, confidence = predict_sentiment(review)
    print(f"{review[:54]:<55} {label:<25} {confidence:.2%}")

Review                                                  Sentiment                 Confidence
------------------------------------------------------------------------------------------
This product is absolutely amazing, works perfectly ou  ⭐⭐⭐⭐⭐ POSITIVE            99.15%
Best purchase I have made this year. Highly recommend   ⭐⭐⭐⭐⭐ POSITIVE            99.21%
Broke after two days. Complete waste of money. Do not   ⭐ NEGATIVE                98.92%
Terrible quality, nothing like the description. Very d  ⭐ NEGATIVE                98.17%
It's okay I guess, nothing special but does the job     ⭐⭐⭐⭐⭐ POSITIVE            95.44%
Not bad for the price but I expected better quality     ⭐⭐⭐⭐⭐ POSITIVE            93.36%
The product is good but delivery took forever           ⭐⭐⭐⭐⭐ POSITIVE            96.97%
I love the design but the battery life is terrible      ⭐ NEGATIVE                96.58%
This distributed systems book completely changed how I  ⭐⭐⭐⭐⭐ POSITIVE            94.96%


In [10]:
flipped_reviews = [
    # Original vs flipped
    ("The battery life is terrible but I love the design", 
     "I love the design but the battery life is terrible"),
    
    ("The product is good but delivery took forever",
     "Delivery took forever but the product is good"),
    
    ("Not bad for the price but I expected better quality",
     "I expected better quality but it's not bad for the price"),
]

print("WORD ORDER SENSITIVITY TEST")
print("=" * 80)
for v1, v2 in flipped_reviews:
    l1, c1 = predict_sentiment(v1)
    l2, c2 = predict_sentiment(v2)
    print(f"\nVersion 1: {v1[:60]}")
    print(f"Result:    {l1} ({c1:.2%})")
    print(f"\nVersion 2: {v2[:60]}")
    print(f"Result:    {l2} ({c2:.2%})")
    changed = "⚠️ CHANGED" if l1 != l2 else "✅ SAME"
    print(f"Prediction: {changed}")
    print("-" * 80)

WORD ORDER SENSITIVITY TEST

Version 1: The battery life is terrible but I love the design
Result:    ⭐⭐⭐⭐⭐ POSITIVE (82.13%)

Version 2: I love the design but the battery life is terrible
Result:    ⭐ NEGATIVE (96.58%)
Prediction: ⚠️ CHANGED
--------------------------------------------------------------------------------

Version 1: The product is good but delivery took forever
Result:    ⭐⭐⭐⭐⭐ POSITIVE (96.97%)

Version 2: Delivery took forever but the product is good
Result:    ⭐⭐⭐⭐⭐ POSITIVE (98.54%)
Prediction: ✅ SAME
--------------------------------------------------------------------------------

Version 1: Not bad for the price but I expected better quality
Result:    ⭐⭐⭐⭐⭐ POSITIVE (93.36%)

Version 2: I expected better quality but it's not bad for the price
Result:    ⭐ NEGATIVE (51.62%)
Prediction: ⚠️ CHANGED
--------------------------------------------------------------------------------
